<a href="https://colab.research.google.com/github/j019/Practical-Machine-Learning/blob/main/Day14/Spacy_Wordnet_for_NLP_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("/content/Airline_sentiment.csv")

In [3]:
df.shape

(14640, 15)

In [4]:
df.columns

Index(['tweet_id', 'airline_sentiment', 'airline_sentiment_confidence',
       'negativereason', 'negativereason_confidence', 'airline',
       'airline_sentiment_gold', 'name', 'negativereason_gold',
       'retweet_count', 'text', 'tweet_coord', 'tweet_created',
       'tweet_location', 'user_timezone'],
      dtype='object')

In [5]:
df['text'].head()

,text
0,@VirginAmerica What @dhepburn said.
1,@VirginAmerica plus you've added commercials t...
2,@VirginAmerica I didn't today... Must mean I n...
3,@VirginAmerica it's really aggressive to blast...
4,@VirginAmerica and it's a really big bad thing...


In [6]:
df['airline_sentiment'].head()

,airline_sentiment
0,neutral
1,positive
2,neutral
3,negative
4,negative


In [7]:
df['airline_sentiment'].unique()

array(['neutral', 'positive', 'negative'], dtype=object)

- It will do all preprocessing steps on text data --> spacy

In [8]:
import spacy
nlp = spacy.load('en_core_web_sm')
# update the stop word list to contain
# most frequent and
# most rare words from the dataset
# then apply the model to text
# ref : https://stackoverflow.com/questions/41170726/add-remove-custom-stop-words-with-spacy

In [9]:
all_text = df['text'].apply(nlp)

In [10]:
len(all_text)

14640

- use tokeniztion algorithm

In [11]:
all_text[:5]

,text
0,"(@VirginAmerica, What, @dhepburn, said, .)"
1,"(@VirginAmerica, plus, you, 've, added, commer..."
2,"(@VirginAmerica, I, did, n't, today, ..., Must..."
3,"(@VirginAmerica, it, 's, really, aggressive, t..."
4,"(@VirginAmerica, and, it, 's, a, really, big, ..."


In [15]:
# is_stop to remove stop words
# lemma_ to extract the lemmatized text
# like_num
# like_email
clean_text = []
for line in all_text:
  clean_line = []
  for token in line:
    if token.is_stop == True: # check for stop words
      pass
    elif token.like_num == True: # check for numeric value
      clean_line.append('num')
    elif token.like_email == True:
      clean_line.append('emailaddr')
    elif token.like_url == True:
      clean_line.append('url')
    elif token.__len__() <=2:
      pass
    elif (token.is_punct == True) or (token.is_quote == True):
      pass
    else:
     # NER
     #if token.ent_type_ == 'PERSON':
     # print(token)
      clean_line.append(token.lemma_.lower())

  clean_text.append(clean_line)

In [16]:
len(clean_text)

14640

In [17]:
clean_text[:5]

[['@virginamerica', '@dhepburn', 'say'],
 ['@virginamerica', 'plus', 'add', 'commercial', 'experience', 'tacky'],
 ['@virginamerica', 'today', 'mean', 'need', 'trip'],
 ['@virginamerica',
  'aggressive',
  'blast',
  'obnoxious',
  'entertainment',
  'guest',
  'face',
  'amp',
  'little',
  'recourse'],
 ['@virginamerica', 'big', 'bad', 'thing']]

# Create dictionary of unique words

In [18]:
clean_text_list = sum(clean_text,[])

# program, plan  --> replace second one with the first one


In [19]:
len(clean_text_list), clean_text_list[:5]

(131217, ['@virginamerica', '@dhepburn', 'say', '@virginamerica', 'plus'])

In [20]:
dictionary = set(clean_text_list)

In [21]:
len(dictionary)

11390

In [22]:
list(dictionary)[:5]

['@_austrian', 'female', 'justgetmehome', 'schoolgirl', 'tonight!great']

# Extra

## Use wordnet to check Synonyms

- wordnet (library) --> words and synonyms or synsets(includes synonyms and antonyms)

## Sample program for checking synonyms and antonyms

In [23]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [24]:
from nltk.corpus import wordnet

In [25]:
syns = wordnet.synsets("program")

In [26]:
syns

[Synset('plan.n.01'),
 Synset('program.n.02'),
 Synset('broadcast.n.02'),
 Synset('platform.n.02'),
 Synset('program.n.05'),
 Synset('course_of_study.n.01'),
 Synset('program.n.07'),
 Synset('program.n.08'),
 Synset('program.v.01'),
 Synset('program.v.02')]

In [27]:
for i in range(len(syns)):
   print(syns[i].lemmas()[0].name())

plan
program
broadcast
platform
program
course_of_study
program
program
program
program


In [28]:
synonyms = []
antonyms = []

for syn in wordnet.synsets("good"):
    for l in syn.lemmas():
        synonyms.append(l.name())
        if l.antonyms():
            antonyms.append(l.antonyms()[0].name())

print(set(synonyms))
print(set(antonyms))

{'honorable', 'ripe', 'right', 'sound', 'estimable', 'dependable', 'dear', 'respectable', 'safe', 'in_effect', 'goodness', 'unspoilt', 'serious', 'just', 'secure', 'proficient', 'skilful', 'commodity', 'beneficial', 'near', 'adept', 'effective', 'salutary', 'full', 'thoroughly', 'skillful', 'undecomposed', 'trade_good', 'practiced', 'good', 'upright', 'soundly', 'in_force', 'expert', 'well', 'unspoiled', 'honest'}
{'ill', 'bad', 'badness', 'evilness', 'evil'}


#### Reference : https://pythonprogramming.net/wordnet-nltk-tutorial/

## Apply wordnet for reducing features in dictionary

In [29]:
word_map={}
rev_map={}
for word in dictionary:
  if word not in rev_map:
    word_map[word]=[]
    for syn in wordnet.synsets(word):
      for l in syn.lemmas():
        if (l.name() != word) and (l.name() in dictionary):
          word_map[word].append(l.name())
          rev_map[l.name()]=word

In [30]:
len(word_map)

9564

In [31]:
len(rev_map)

2099

In [32]:
rev_map

{'outstanding': 'greatest',
 'junk': 'dust',
 'scatter': 'dust',
 'sprinkle': 'dust',
 'dot': 'points',
 'draw': 'forced',
 'catch': 'grab',
 'cart': 'haul',
 'drag': 'haul',
 'mile': 'miles',
 'crook': 'criminal',
 'twist': 'device',
 'play': 'working',
 'go': 'adam',
 'spell': 'piece',
 'tour': 'circuit',
 'bout': 'turn',
 'round': 'beats',
 'act': 'working',
 'routine': 'mundane',
 'number': 'numbered',
 'bit': 'piece',
 'reverse': 'turnaround',
 'grow': 'farms',
 'release': 'waiver',
 'plow': 'deals',
 'plough': 'turn',
 'wrench': 'turn',
 'rick': 'turn',
 'flex': 'turn',
 'sour': 'working',
 'work': 'working',
 'regard': 'respect',
 'assuage': 'relieved',
 'appease': 'pacify',
 'action': 'processing',
 'process': 'working',
 'gang': 'ring',
 'crowd': 'herded',
 'bunch': 'cluster',
 'hope': 'promise',
 'assure': 'assuring',
 'predict': 'calls',
 'call': 'calls',
 'anticipate': 'calls',
 'repair': 'revive',
 'fix': 'makes',
 'doctor': 'dr.',
 'reinstate': 'restore',
 'chicken': 'yel